# 00 · Preparación del entorno en Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StevenFAP1815/project-reconstruction-3d/blob/main/notebooks/00_setup_colab.ipynb)

Este notebook deja lista una sesión de Colab para el proyecto:

1. Monta Google Drive.
2. Descarga (o actualiza) el código del repositorio y lo hace importable.
3. Instala las dependencias fijadas en `requirements-colab.txt`.
4. Crea la carpeta `Mi unidad/project-reconstruction-3d/` con su estructura.
5. Comprueba la GPU y registra las versiones del entorno.
6. Extrae las fotos de cada objeto al disco local de la sesión.

**Antes de ejecutar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU*.

Las celdas 1 a 5 se copian al inicio de cada notebook del pipeline, porque
cada sesión de Colab empieza vacía.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Código del repositorio

GitHub es la fuente oficial del código. Se clona en `/content` (disco de la
sesión) y se añade `src/` al path para poder hacer `import recon3d`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/StevenFAP1815/project-reconstruction-3d.git"
REPO_DIR = Path("/content/project-reconstruction-3d")
BRANCH = "main"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True
    )

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

print(subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

## 3. Dependencias fijadas

In [ ]:
%pip install -q -r /content/project-reconstruction-3d/requirements-colab.txt

## 4. Estructura de carpetas en Drive

`create()` usa `exist_ok=True`: si las carpetas ya existen no hace nada, así
que se puede ejecutar siempre sin riesgo de borrar resultados.

In [ ]:
from recon3d.paths import OBJECTS, get_paths

paths = get_paths()  # en Colab: /content/drive/MyDrive/project-reconstruction-3d
for folder in paths.create():
    print(folder)

## 5. GPU, semilla y versiones del entorno

In [ ]:
import json

from recon3d.reproducibility import set_seed
from recon3d.results import environment_info

SEED = 42
set_seed(SEED)

env = environment_info()
print(json.dumps(env, indent=2))
if env["gpu"] is None:
    print("AVISO: no hay GPU. Cambiar el tipo de entorno de ejecución a GPU.")

## 6. Fotos de los objetos

Las fotos se suben a Drive como un zip por objeto en `data/zips/`
(generados en local con `python scripts/make_data_zips.py`). Aquí se extraen al
disco local de la sesión (`/content/data`), que es mucho más rápido de leer
que Drive.

In [ ]:
from recon3d.dataset import extract_object_zip

LOCAL_DATA = Path("/content/data")

for name in (*OBJECTS, "calibration"):
    zip_path = paths.data_zip(name)
    if not zip_path.exists():
        print(f"[falta] {zip_path}")
        continue
    images = extract_object_zip(zip_path, LOCAL_DATA / name)
    print(f"[ok] {name}: {len(images)} imágenes en {LOCAL_DATA / name}")

## Siguiente paso

Con el entorno listo, se continúa con los notebooks del pipeline
(`01_calibration`, `02_sfm_colmap`, ...). Cada corrida guarda sus salidas en
`paths.run_dir(objeto, metodo)` y registra su `meta.json` con
`recon3d.results.write_meta`.

Para llevar los resultados al repositorio local: descargar
`Mi unidad/project-reconstruction-3d/results/` y pegar su contenido en
`results/` del repositorio. Git solo versiona los `meta.json`.